###Build Results Fact
1. Read silver results tabel
2. Read silver sprints table
3. Add new column session_type with values RACE or SPRINT
4. UNION results and sprints
5. Derive additional columns
-     is_win -> indicates that the driver won the race
-     is_podium -> indicates that the driver scored a podium result (1,2,3)
-     has_points -> indicates that the driver has scored points
6. Write the transformed data to gold fact|_session_results table

In [0]:
%run ../00-Common/01.environment-config

In [0]:
target_table = f"{catalog_name}.{gold_schema}.fact_session_results"

In [0]:
from pyspark.sql import functions as f

In [0]:
results_df = (
    spark.table(f"{catalog_name}.{silver_schema}.results")
        .withColumn("session_type",f.lit("RACE"))
        .drop("race_name","race_date","current_timestamp","source_file")
)
 

In [0]:
sprints_df = (
    spark.table(f"{catalog_name}.{silver_schema}.sprints")
        .withColumn("session_type",f.lit("SPRINTS"))
        .drop("race_name","race_date","current_timestamp","source_file")
)

In [0]:
results_sprints_df = results_df.unionByName(sprints_df)

In [0]:
fact_session_results_df = (
    results_sprints_df
        .withColumn("is_win", f.col("finish_position") == 1)
        .withColumn("is_podimun", f.col("finish_position").between (1,3))
        .withColumn("has_points", f.col("points") > 0)
)

In [0]:
(
    fact_session_results_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
)

In [0]:
display(spark.table(target_table).filter("season = 2025"))